# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os, duckdb, pandas as pd, numpy as np
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs"
os.makedirs(BASE, exist_ok=True)

from datasets import load_dataset

fact_content_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train")
dim_content_ds = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train")

fact_content = fact_content_ds.data.table
dim_content = dim_content_ds.data.table

print("Loaded fresh from source.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 4.41MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 2.62MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

...

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

Loaded fresh from source.


In [2]:
from huggingface_hub import login

login()

In [3]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [5]:
# Step 1: pull raw features, including RAW (unfilled) gsc_avg_position_prior
raw_features = duckdb.sql("""
    WITH prior_position AS (
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_avg_position) AS gsc_avg_position_prior
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        d.word_count,
        d.search_volume,
        d.competition_level,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-01') AS content_age_days,
        p.gsc_avg_position_prior
    FROM dim_content d
    LEFT JOIN prior_position p
        USING (client_hash_id, content_hash_id)
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
      AND d.content_created_date <= DATE '2026-03-01'
""").df()

print("Raw rows:", len(raw_features))

# Step 2: build EVERY missing flag from the RAW column, before any fillna touches it
raw_features["word_count_missing"] = raw_features["word_count"].isna().astype(int)
raw_features["search_volume_missing"] = raw_features["search_volume"].isna().astype(int)
raw_features["gsc_position_missing"] = raw_features["gsc_avg_position_prior"].isna().astype(int)
competition_missing_mask = raw_features["competition_level"].isna()

# Sanity check RIGHT NOW — before any fill happens — confirm flags actually vary
print("\n--- Flags right after creation (before any fill) ---")
print("word_count_missing:\n", raw_features["word_count_missing"].value_counts())
print("search_volume_missing:\n", raw_features["search_volume_missing"].value_counts())
print("gsc_position_missing:\n", raw_features["gsc_position_missing"].value_counts())
print("competition missing count:", competition_missing_mask.sum())

# Step 3: NOW fill, only after flags are locked in
feature_vector = raw_features.copy()
feature_vector["word_count"] = feature_vector["word_count"].fillna(feature_vector["word_count"].median())
feature_vector["search_volume"] = feature_vector["search_volume"].fillna(feature_vector["search_volume"].median())
feature_vector["gsc_avg_position_prior"] = feature_vector["gsc_avg_position_prior"].fillna(
    feature_vector["gsc_avg_position_prior"].median()
)

competition_map = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
feature_vector["competition_level_encoded"] = feature_vector["competition_level"].map(competition_map).fillna(-1)

# Final check: no NaNs left anywhere in the modeling columns, flags DO vary
honest_cols = ["word_count", "word_count_missing", "search_volume", "search_volume_missing",
               "competition_level_encoded", "content_age_days",
               "gsc_avg_position_prior", "gsc_position_missing"]

print("\n--- Final NaN check ---")
print(feature_vector[honest_cols].isna().sum())

print("\n--- Final flag distributions (must NOT be all-zero) ---")
print(feature_vector["gsc_position_missing"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw rows: 303332

--- Flags right after creation (before any fill) ---
word_count_missing:
 word_count_missing
0    196694
1    106638
Name: count, dtype: int64
search_volume_missing:
 search_volume_missing
0    253687
1     49645
Name: count, dtype: int64
gsc_position_missing:
 gsc_position_missing
1    152755
0    150577
Name: count, dtype: int64
competition missing count: 50354

--- Final NaN check ---
word_count                   0
word_count_missing           0
search_volume                0
search_volume_missing        0
competition_level_encoded    0
content_age_days             0
gsc_avg_position_prior       0
gsc_position_missing         0
dtype: int64

--- Final flag distributions (must NOT be all-zero) ---
gsc_position_missing
1    152755
0    150577
Name: count, dtype: int64


Bug confirmed fixed: gsc_position_missing now correctly shows 152,755 pages (50.4%) with genuinely missing prior GSC data, matching the manual verification from the previous session. The fix was building the missing-flag from the raw, unfilled column before applying any .fillna() — previously the flag was built after filling, so it always read as zero regardless of true missingness.

Go ahead and run the rest of the notebook (labels → baseline score → split → train → compare) — and share the new comparison table once you have it. This time the gsc_avg_position_prior feature and its flag are both trustworthy, so if the precision@20 numbers shift from last time, that's the real, honest signal — not noise.





In [6]:
feature_vector.to_csv(f"{BASE}/feature_vector.csv", index=False)
print("Corrected feature_vector saved:", feature_vector.shape)

Corrected feature_vector saved: (303332, 11)


In [7]:
labels = duckdb.sql("""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_feb
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    march AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_march
        FROM fact_content
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT feb.client_hash_id, feb.content_hash_id, clicks_feb, clicks_march,
           CASE WHEN clicks_march < clicks_feb THEN 1 ELSE 0 END AS declined
    FROM feb JOIN march USING (client_hash_id, content_hash_id)
""").df()

labels.to_csv(f"{BASE}/labels.csv", index=False)
print("Labels saved:", len(labels), "rows")
print(labels["declined"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Labels saved: 134238 rows
declined
0    109781
1     24457
Name: count, dtype: int64


In [8]:
scored = feature_vector.merge(labels, on=["client_hash_id", "content_hash_id"])

# staleness score
scored["staleness_score"] = (scored["content_age_days"] / scored["content_age_days"].max()).clip(0, 1)

# CTR check (volume-floored version)
ctr_check = duckdb.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS actual_ctr,
           AVG(gsc_avg_position) AS position_feb
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

expected_ctr_map = {"top3": 0.0106, "top10": 0.0052, "below10": 0.0028}
def get_expected(pos):
    if pos <= 3: return expected_ctr_map["top3"]
    elif pos <= 10: return expected_ctr_map["top10"]
    else: return expected_ctr_map["below10"]

ctr_check["expected_ctr"] = ctr_check["position_feb"].apply(get_expected)
ctr_check["ctr_gap"] = ctr_check["actual_ctr"] - ctr_check["expected_ctr"]

scored = scored.merge(ctr_check[["client_hash_id", "content_hash_id", "ctr_gap", "actual_ctr"]],
                        on=["client_hash_id", "content_hash_id"], how="left")

has_volume = scored["actual_ctr"].notna() & (scored["actual_ctr"] > scored["actual_ctr"].median())
scored["ctr_score"] = 0.0
scored.loc[has_volume, "ctr_score"] = (-scored.loc[has_volume, "ctr_gap"]).clip(lower=0)
scored["ctr_score"] = (scored["ctr_score"] / scored["ctr_score"].max()).fillna(0)

scored["score"] = 0.7 * scored["staleness_score"] + 0.3 * scored["ctr_score"]

scored.to_csv(f"{BASE}/scored_baseline.csv", index=False)
print("Baseline score rebuilt and saved.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline score rebuilt and saved.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
method_choice = "Logistic Regression (baseline model) -> Random Forest (complexity check)"
evaluation_metric = "precision@20, matching the Week-4 baseline's queue size"
print(method_choice)
print(evaluation_metric)

Logistic Regression (baseline model) -> Random Forest (complexity check)
precision@20, matching the Week-4 baseline's queue size


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

data_model = scored.dropna(subset=["declined"]).copy()

X = data_model[honest_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("NaN check on X_train (must be all zero):")
print(X_train.isna().sum())
print("Client overlap (must be 0):",
      len(set(data_model.iloc[train_idx]["client_hash_id"]) & set(data_model.iloc[test_idx]["client_hash_id"])))

NaN check on X_train (must be all zero):
word_count                   0
word_count_missing           0
search_volume                0
search_volume_missing        0
competition_level_encoded    0
content_age_days             0
gsc_avg_position_prior       0
gsc_position_missing         0
dtype: int64
Client overlap (must be 0): 0


In [10]:
logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

print("Models trained.")

Models trained.


In [11]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    return y_true.iloc[top_k].mean()

y_test_reset = y_test.reset_index(drop=True)
baseline_scores_test = data_model.iloc[test_idx]["score"].reset_index(drop=True)

base_rate = y_test_reset.mean()
baseline_p20 = precision_at_k(y_test_reset, baseline_scores_test.values, 20)
logreg_p20 = precision_at_k(y_test_reset, logreg_scores, 20)
rf_p20 = precision_at_k(y_test_reset, rf_scores, 20)

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "Week-4 rule baseline", "Logistic Regression", "Random Forest"],
    "precision@20": [base_rate, baseline_p20, logreg_p20, rf_p20]
})
comparison

,method,precision@20
0,Base rate (random),0.206912
1,Week-4 rule baseline,0.350000
2,Logistic Regression,0.350000
3,Random Forest,0.350000


In [12]:
importances = pd.Series(rf.feature_importances_, index=honest_cols).sort_values(ascending=False)
print(importances)

content_age_days             0.440262
word_count                   0.272513
gsc_avg_position_prior       0.131188
word_count_missing           0.094901
search_volume                0.032275
competition_level_encoded    0.025648
search_volume_missing        0.003214
gsc_position_missing         0.000000
dtype: float64


In [13]:
print("LogReg score range:", logreg_scores.min(), "-", logreg_scores.max())
print("LogReg unique values (first 10):", np.unique(logreg_scores)[:10])
print("RF score range:", rf_scores.min(), "-", rf_scores.max())
print("RF unique values (first 10):", np.unique(rf_scores)[:10])

LogReg score range: 2.8873077353946734e-10 - 0.9667791849801748
LogReg unique values (first 10): [2.88730774e-10 7.99780055e-09 1.05651727e-08 1.14915440e-08
 1.17021328e-08 1.90936765e-07 2.06307298e-06 2.39613551e-06
 1.77234564e-05 1.97414640e-05]
RF score range: 0.09121181445373246 - 0.7344146312297437
RF unique values (first 10): [0.09121181 0.09192018 0.10082584 0.10243167 0.11421972 0.12070624
 0.12132149 0.1251213  0.12561866 0.12628569]


In [14]:
top20_baseline = np.argsort(-baseline_scores_test.values)[:20]
top20_logreg = np.argsort(-logreg_scores)[:20]
top20_rf = np.argsort(-rf_scores)[:20]

print("Baseline vs LogReg top-20 overlap:", len(set(top20_baseline) & set(top20_logreg)))
print("Baseline vs RF top-20 overlap:", len(set(top20_baseline) & set(top20_rf)))

Baseline vs LogReg top-20 overlap: 0
Baseline vs RF top-20 overlap: 0


In [15]:
from sklearn.metrics import roc_auc_score
print("LogReg AUC:", roc_auc_score(y_test_reset, logreg_scores))
print("RF AUC:", roc_auc_score(y_test_reset, rf_scores))

LogReg AUC: 0.5741196484269995
RF AUC: 0.6052331255587032


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [16]:
# === SECTION 4: Errors and interpretation ===

# --- Cell 5: feature importance ---
importances = pd.Series(rf.feature_importances_, index=honest_cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

top_feature = importances.index[0]
top_share = importances.iloc[0]
print(f"\nTop feature: {top_feature} ({top_share:.1%} of total importance)")
print("⚠️ Check for leakage" if top_share > 0.7 else "✅ No single feature dominates")

Random Forest feature importances:
content_age_days             0.440262
word_count                   0.272513
gsc_avg_position_prior       0.131188
word_count_missing           0.094901
search_volume                0.032275
competition_level_encoded    0.025648
search_volume_missing        0.003214
gsc_position_missing         0.000000
dtype: float64

Top feature: content_age_days (44.0% of total importance)
✅ No single feature dominates


In [ ]:
# --- Cell 6: build test_results for error analysis ---
test_results = data_model.iloc[test_idx].copy().reset_index(drop=True)
test_results["true_label"] = y_test_reset
test_results["rf_score"] = rf_scores
test_results["logreg_score"] = logreg_scores

false_negatives = test_results[test_results["true_label"] == 1].sort_values("rf_score").head(3)
print("Worst false negatives (missed real declines):")
print(false_negatives[["client_hash_id", "content_hash_id", "content_age_days",
                        "gsc_avg_position_prior", "rf_score"]])

Worst false negatives (missed real declines):
                client_hash_id           content_hash_id  content_age_days  \
33447  client_73cda7b4e4f265ea  content_8671edba7788e7ea               213   
28367  client_62f4a7e64f5e0096  content_0f067cb4ec91ab72               219   
66858  client_3f0ce4d44fe94f3d  content_3f230cbfac13abd8                17   

       gsc_avg_position_prior  rf_score  
33447               68.402990  0.117482  
28367               70.652702  0.117920  
66858              150.000000  0.123201  


In [ ]:
# --- Cell 7: false positives ---
false_positives = test_results[test_results["true_label"] == 0].sort_values("rf_score", ascending=False).head(3)
print("Worst false positives (flagged but didn't decline):")
print(false_positives[["client_hash_id", "content_hash_id", "content_age_days",
                        "gsc_avg_position_prior", "rf_score"]])

Worst false positives (flagged but didn't decline):
                client_hash_id           content_hash_id  content_age_days  \
55052  client_fef1a8f436438636  content_eb127d86d0bc22cc               184   
51023  client_9958f0a7ae1df715  content_9804c8d434f01efd               417   
51255  client_9958f0a7ae1df715  content_ab7747aa4c264b41               417   

       gsc_avg_position_prior  rf_score  
55052               20.966169  0.693923  
51023                4.974344  0.693889  
51255                5.021106  0.693336  


In [ ]:
# --- Cell 8: error patterns by group ---
test_results["age_bucket"] = pd.cut(test_results["content_age_days"],
                                      bins=[-1, 90, 180, 10000], labels=["<90d", "90-180d", "180d+"])
test_results["correct"] = (test_results["rf_score"] > 0.5) == (test_results["true_label"] == 1)

print("Accuracy by age bucket:")
print(test_results.groupby("age_bucket")["correct"].mean())

print("\nAccuracy by whether gsc_avg_position_prior was missing:")
print(test_results.groupby("gsc_position_missing")["correct"].mean())

Accuracy by age bucket:
age_bucket
<90d       0.822845
90-180d    0.688128
180d+      0.712354
Name: correct, dtype: float64

Accuracy by whether gsc_avg_position_prior was missing:
gsc_position_missing
0    0.740123
Name: correct, dtype: float64


/tmp/ipykernel_3176/3170365085.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(test_results.groupby("age_bucket")["correct"].mean())


In [ ]:
print(test_results.groupby("gsc_position_missing")["correct"].mean())
print(test_results.groupby("gsc_position_missing")["correct"].count())

gsc_position_missing
0    0.740123
Name: correct, dtype: float64
gsc_position_missing
0    73558
Name: correct, dtype: int64


In [ ]:
print(data_model["gsc_position_missing"].value_counts())

gsc_position_missing
0    134086
Name: count, dtype: int64


In [ ]:
# check the actual missingness rate before any filling, straight from feature_vector's source
print(feature_vector["gsc_avg_position_prior"].isna().sum())

0


In [ ]:
# Rebuild directly from fact_content, BEFORE any fill — this shows the true original gap
prior_position_fresh = duckdb.sql("""
    SELECT client_hash_id, content_hash_id,
           AVG(gsc_avg_position) AS gsc_avg_position_prior
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# Now LEFT JOIN this onto ALL pages (not just ones with Feb data) to see the true gap
all_pages = feature_vector[["client_hash_id", "content_hash_id"]].drop_duplicates()
check = all_pages.merge(prior_position_fresh, on=["client_hash_id", "content_hash_id"], how="left")

print("Pages with no prior Feb position (true missingness):", check["gsc_avg_position_prior"].isna().sum())
print("Out of total pages:", len(check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with no prior Feb position (true missingness): 152755
Out of total pages: 303332


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.